# Extract Item 7, MD&A, from your 100 10-Ks. Put MD&A in your dataframe in one column. Run all of your code on the MD&A and put in Dataframe. (your dataframe will contain data for the entire 10-K, and data for the MD&A). Save your dataframe as a pickle object so you don’t have to start from scratch anymore!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Import the Required Libraries

In [ ]:
import re
from bs4 import BeautifulSoup
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize
import pandas as pd
import os

## Syllabe Count

In [ ]:
# Set the Drive path of the pronunciation dictionary; adjust it to your own folder.
CMUDICT_PATH = "YOUR_DATA_DIRECTORY"

# Create an empty dictionary for words and their pronunciation symbols.
cmudict = {}
with open(CMUDICT_PATH, encoding="latin-1") as f:
  for line in f:
    if line.startswith(";;;"):
      continue
    cols = line.split()
    if not cols:
      continue
    word = cols[0]
    if "(" in word:
      word = word[:word.index("(")]
    cmudict.setdefault(word.lower(), cols[1:])

print(len(cmudict))

# Count Syllables and Identify Complex Words

In [ ]:
# Estimate syllables from spelling when a word is missing from the dictionary.
def countVowelGroups(word):
  word = word.lower()
  groups = re.findall(r"[aeiouy]+", word)
  numSyllables = len(groups)
  if word.endswith("e") and not word.endswith(("le", "ee", "ye")) and numSyllables > 1:
    numSyllables -= 1
  return max(numSyllables, 1)

syllableCache = {}

# Return the syllable count using the cache, dictionary, or spelling fallback.
def getSyllables(words):
  word = words.lower()
  if word in syllableCache:
    return syllableCache[word]

  phonemes = cmudict.get(word)
  if phonemes is None:
    syllables = countVowelGroups(word)
  else:
    numSyllables = sum(1 for p in phonemes if p[-1].isdigit())
    syllables = numSyllables

  syllableCache[word] = syllables
  return syllables

def isComplexWord(word):
  return getSyllables(word) > 2

print(getSyllables("Kevin"), isComplexWord("Kevin"))
print(getSyllables("readability"), isComplexWord("readability"))

## Identify Future-Oriented Sentences

In [ ]:
# Define the lowercase keyword set used to flag sentences.
FUTURE_WORDS = {"will", "shall", "may", "might", "should"}

# Return whether the sentence contains one of the selected expressions.
def isFuture(sentence, words=None):
  if words is None:
    words = [w.lower() for w in word_tokenize(sentence)]

  for word in words:
    if word in FUTURE_WORDS:
      return True

  # Examine adjacent token pairs without going beyond the list.
  # "going to" is two tokens; NLTK also splits "gonna" into "gon" + "na"
  for i in range(len(words) - 1):
    if words[i] == "going" and words[i + 1] == "to":
      return True
    if words[i] == "gon" and words[i + 1] == "na":
      return True

  return False


text = "NLTK is gonna be a great NLP toolkit. It makes processing text easy! Revenue will grow."
for s in sent_tokenize(text):
  print(isFuture(s), "|", s)

### Sentiment WordLoad Sentiment Dictionaries and Analyze Text

In [ ]:
# Read the Loughran–McDonald CSV from Drive; this file must exist at the given path.
lm = pd.read_csv("YOUR_DATA_DIRECTORY")
# Convert dictionary words to lowercase strings to match the analysis tokens.
lm["Word"] = lm["Word"].astype(str).str.lower()

# Keep dictionary words
positive  = set(lm.loc[lm["Positive"]  > 0, "Word"])
negative  = set(lm.loc[lm["Negative"]  > 0, "Word"])
litigious = set(lm.loc[lm["Litigious"] > 0, "Word"])

print(len(positive), len(negative), len(litigious))

# Accept lowercase alphabetic tokens with optional straight apostrophes; this is the analysis word definition.
WORD_RE = re.compile(r"^[a-z][a-z']*$")

# Return raw counts for the complete text and its flagged sentences.
def analyzeDocument(text):
  stats = {
      "sentenceCount": 0, "future_sentenceCount": 0,
      "wordCount": 0,     "future_wordCount": 0,
      "complexWordCount": 0, "future_complexWordCount": 0,
      "syllableCount": 0, "future_syllableCount": 0,
      "PosWordCount": 0,  "future_PosWordCount": 0,
      "NegWordCount": 0,  "future_NegWordCount": 0,
      "LitWordCount": 0,  "future_LitWordCount": 0,
  }

  for sentence in sent_tokenize(text):
    tokens = [w.lower() for w in word_tokenize(sentence)]
    words = [w for w in tokens if WORD_RE.match(w)]
    if not words:
      continue

    future = isFuture(sentence, tokens)

    stats["sentenceCount"] += 1
    if future:
      stats["future_sentenceCount"] += 1

    for word in words:
      syl = getSyllables(word)
      complexWord = syl > 2

      stats["wordCount"] += 1
      stats["syllableCount"] += syl
      if complexWord:
        stats["complexWordCount"] += 1
      if word in positive:
        stats["PosWordCount"] += 1
      elif word in negative:
        stats["NegWordCount"] += 1
      elif word in litigious:
        stats["LitWordCount"] += 1

      if future:
        stats["future_wordCount"] += 1
        stats["future_syllableCount"] += syl
        if complexWord:
          stats["future_complexWordCount"] += 1
        if word in positive:
          stats["future_PosWordCount"] += 1
        elif word in negative:
          stats["future_NegWordCount"] += 1
        elif word in litigious:
          stats["future_LitWordCount"] += 1

  return stats

## Calculate Readability Scores and Ratios
### Convert the raw counts into analysis columns. Calculate the FOG index, sentiment ratios, and future-oriented sentence and word ratios.





In [ ]:
def fogIndex(wordCount, sentenceCount, complexWordCount):
  if wordCount == 0 or sentenceCount == 0:
    return 0.0
  return 0.4 * ((wordCount / sentenceCount) + 100.0 * (complexWordCount / wordCount))

def rate(numerator, denominator):
  if denominator == 0:
    return 0.0
  return numerator / denominator

def makeColumns(s, prefix=""):
  cols = {}
  for who in ["", "future_"]:
    wordCount        = s[who + "wordCount"]
    sentenceCount    = s[who + "sentenceCount"]
    complexWordCount = s[who + "complexWordCount"]
    syllableCount    = s[who + "syllableCount"]

    cols[prefix + who + "wordCount"]        = wordCount
    cols[prefix + who + "sentenceCount"]    = sentenceCount
    cols[prefix + who + "complexWordCount"] = complexWordCount
    cols[prefix + who + "PosWordCount"]     = s[who + "PosWordCount"]
    cols[prefix + who + "NegWordCount"]     = s[who + "NegWordCount"]
    cols[prefix + who + "LitWordCount"]     = s[who + "LitWordCount"]
    cols[prefix + who + "PosRate"]          = rate(s[who + "PosWordCount"], wordCount)
    cols[prefix + who + "NegRate"]          = rate(s[who + "NegWordCount"], wordCount)
    cols[prefix + who + "LitRate"]          = rate(s[who + "LitWordCount"], wordCount)
    cols[prefix + who + "complexRate"]      = rate(complexWordCount, wordCount)
    cols[prefix + who + "wordsPerSentence"] = rate(wordCount, sentenceCount)
    cols[prefix + who + "syllablesPerWord"] = rate(syllableCount, wordCount)
    cols[prefix + who + "FOG"]              = fogIndex(wordCount, sentenceCount, complexWordCount)

  cols[prefix + "futureSentenceRate"] = rate(s["future_sentenceCount"], s["sentenceCount"])
  cols[prefix + "futureWordRate"]     = rate(s["future_wordCount"], s["wordCount"])
  return cols

## Extract the First Document and Clean HTML
### Select the first DOCUMENT block, read its form type, and extract its TEXT content. BeautifulSoup removes selected content and keeps text with paragraph breaks.

In [ ]:
# Extract the first DOCUMENT block; this alone does not guarantee that it is a 10-K.
def firstDocument(submission):
    # Find the first DOCUMENT block; ignore case and allow matches across line breaks.
    block = re.search(r"<DOCUMENT>(.*?)</DOCUMENT>", submission, re.I | re.S)
    # Check whether a DOCUMENT block was found.
    if block is not None:
        return block.group(1)
    # Use this alternative when the preceding condition is false.
    else:
        return ""

# Return the first document form type or an empty string.
def getFormType(submission):
  # Get the first document from the submission.
    document = firstDocument(submission)
    # Read the form type, stopping at a line break or the next tag.
    match = re.search(r"<TYPE>[ \t]*([^\r\n<]+)", document, re.I)
    if match is not None:
        return match.group(1).strip()
    else:
        return ""

# Return TEXT contents, or the first document when TEXT tags are missing.
def getDocumentHtml(submission):
    document = firstDocument(submission)
    # Find the TEXT content within the selected document.
    match = re.search(r"<TEXT>(.*?)</TEXT>", document, re.I | re.S)
    if match is not None:
        return match.group(1)
    else:
        return document

# Clean selected HTML contents and normalize whitespace.
def cleanSECText(submission):

    # Extract the content to be cleaned.
    html_content = getDocumentHtml(submission)
    # Parse the HTML with the built-in HTML parser.

    soup = BeautifulSoup(html_content, "html.parser")
    text = soup.get_text()

    return text.strip()

# Define the Item 7 Start and End Patterns

In [ ]:
# Allow spaces or tabs and at most one newline between heading elements.
GAP = r"[ \t]*\n?[ \t]*"

# Allow an optional period, colon, closing parenthesis, or dash after the item number.

PUNCT = r"[.:)\-\u2013\u2014]?"

# Match a line-start Item 7 heading followed by management, with the allowed spacing and punctuation.
start_pattern = r"^[ \t]*item" + GAP + r"7" + GAP + PUNCT + r"\s{0,3}management"

# Match a line-start Item 7A or Item 8 heading; prevent matches such as Item 80.
end_pattern = r"^[ \t]*item" + GAP + r"(?:7A|8)(?![0-9A-Za-z])"

# Ignore case and let the start-of-line marker match every line.
flags = re.IGNORECASE | re.MULTILINE

# Prepare the Item 7 start-heading pattern for repeated searches.
ITEM7_RE = re.compile(start_pattern, flags)

# Prepare the ending pattern; despite its name it matches Item 7A or Item 8.
ITEM7A_RE = re.compile(end_pattern, flags)

# Extract MD&A and Assign a Basic Status

In [ ]:
# Return the longest Item 7 candidate from the cleaned text.
def getMDA(cleanText):
    # Start with an empty MD&A candidate.
    longest_text = ""

    # Try each matching Item 7 heading, including possible contents entries.
    for heading in ITEM7_RE.finditer(cleanText):
        start = heading.start()
        # Find the first matching ending heading after this starting position.
        ending = ITEM7A_RE.search(cleanText, start + 1)

        if ending is None:
            end = len(cleanText)
        else:
            end = ending.start()
        # Slice the candidate text, including Item 7 but excluding the ending heading.
        candidate = cleanText[start:end]

        # Keep this candidate only if it is longer than the previous one.
        if len(candidate) > len(longest_text):
          # Replace the saved candidate with the longer text.
            longest_text = candidate

    return longest_text.strip()


# Perform basic extraction checks; cleanText is an unused parameter retained for compatibility.
def mdaStatus(mdaText, cleanText, formType):
    if not formType.upper().startswith("10-K"):
        return "not an annual report"

    if not mdaText:
        return "no Item 7 heading"

    return "valid"

# Collect the Input File Paths

In [ ]:
# Set the input and output folder path in Google Drive.
folder = "YOUR_DATA_DIRECTORY"

files = sorted(
    os.path.join(folder, name)
    for name in os.listdir(folder)
    if os.path.isfile(os.path.join(folder, name))
)

# Show the list length; the label assumes that all included files are TXT.
print("Number of TXT files found:", len(files))

# Stop if there are no input files.
if len(files) == 0:
    raise FileNotFoundError("No files found. Please check the folder path.")

# Check whether the number of listed files differs from the expected 100.
if len(files) != 100:
    print("Note: All files found in the folder will be analyzed.")

# Analyze Each Filing and Build the DataFrame

In [ ]:
# Create an empty list to collect one result dictionary per input file.
rows = []

# Process the full file paths one at a time.
for path in files:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        submission = f.read()

     # Read the actual type from the first document, rather than trusting the file name.
    formType = getFormType(submission)
     # Clean the selected document and retain its full text.
    cleanText = cleanSECText(submission)
    # Extract an MD&A candidate from the cleaned full text.
    mdaText = getMDA(cleanText)
    # Apply the basic form-type and nonempty-text checks.
    status = mdaStatus(mdaText, cleanText, formType)

    if status != "valid":
        mdaText = ""

# Create the file-information dictionary that will become one DataFrame row.
    row = {
        "fileName": os.path.basename(path),
        "formType": formType,
        "cleanText": cleanText,
        "mdaText": mdaText,
        "mdaStatus": status
    }


    # Calculate raw counts for the full selected document. whole_stats = analyzeDocument(cleanText)
    whole_result = makeColumns(whole_stats)
    row.update(whole_result)

    # Calculate the same raw counts for MD&A.
    mda_stats = analyzeDocument(mdaText)
    mda_result = makeColumns(mda_stats)

    # Read each column name and value from the MD&A analysis dictionary.
    for column, value in mda_result.items():
        row["mda_" + column] = value
    # Add this completed file result to the result list.
    rows.append(row)

df = pd.DataFrame(rows)

print("DataFrame shape:", df.shape)
display(df["mdaStatus"].value_counts())
display(df.head())

# Save the Full DataFrame

In [ ]:
# Save the complete DataFrame, including both full text columns, to the input folder.
df.to_pickle(os.path.join(folder, "tenk_mda_analysis.pkl"))

csv_df = df.drop(columns=["cleanText", "mdaText"])

csv_path = os.path.join(folder, "tenk_mda_analysis.csv")
csv_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Pickle saved.")
print("CSV saved:", csv_path)

## Check the contents of the pickle file


In [ ]:
folder = "YOUR_DATA_DIRECTORY"

df = pd.read_pickle(os.path.join(folder, "tenk_mda_analysis.pkl"))

print("DataFrame shape:", df.shape)
display(df.head())